# Experiment 2 — LR: X → Y vs X + C → Y

The **one-thing test**: a single, identical logistic regression, with the
only change being the addition of the semantic-bottleneck features C
(the Ideas supporting each sentence). Same 60/40 **document-level** split,
same seed, same hyperparameters.

- **A (control):** TF-IDF sentence features → label
- **B (experiment):** TF-IDF sentence features **+ C multi-hot over the
  8,706-idea space** → label

C is label-conditioned by construction (QSBC Phase 1: the teacher is shown Y
and produces the general reasoning principles). This makes B the **ceiling**
for the pipeline: it measures whether the idea layer carries signal worth
extracting (~+10pt is the target). The deployed gain = ceiling × Stage-A
extraction quality.

Metrics: **accuracy + macro-F1** on the held-out test set. Per-class report
for B, plus the exact per-idea → label attribution weights (linear probe).


In [1]:
# 1. Setup + fetch committed inputs
import os
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
os.chdir('/content/qsbc')
print('cwd:', os.getcwd())
!pip install -q pandas scikit-learn


cwd: /content/qsbc


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('results/exp1/samples_2858.csv')
df = df.drop_duplicates(subset=['sentence_id']).reset_index(drop=True)
ideas = pd.read_csv('results/exp1/sample_ideas.csv')

# sentence -> list of idea_ids (the supporting C set)
grp = ideas.groupby('sentence_id')['idea_id'].apply(list).to_dict()
df['ideas'] = df['sentence_id'].map(grp).fillna('').apply(
    lambda x: x if isinstance(x, list) else [])

print('samples:', len(df), '| with >=1 idea:', int((df['ideas'].str.len() > 0).sum()))
print('idea space size:', ideas['idea_id'].nunique())
print('ideas per sample (mean):', round(df['ideas'].str.len().mean(), 2))


samples: 2858 | with >=1 idea: 2782
idea space size: 8706
ideas per sample (mean): 4.05


In [3]:
from sklearn.model_selection import train_test_split

SEED = 42
docs = df['doc_key'].unique()
train_docs, test_docs = train_test_split(docs, test_size=0.4,
                                          random_state=SEED)
train = df[df['doc_key'].isin(train_docs)].reset_index(drop=True)
test  = df[df['doc_key'].isin(test_docs)].reset_index(drop=True)
print('doc overlap:', len(set(train_docs) & set(test_docs)))
print('train:', len(train), '| test:', len(test))
print('train classes:', train['label'].nunique(), '| test classes:', test['label'].nunique())


doc overlap: 0
train: 1748 | test: 1110
train classes: 12 | test classes: 12


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack

# X features: TF-IDF over sentences (identical to exp1)
tf = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                     stop_words='english')
Xtr = tf.fit_transform(train['sentence'])
Xte = tf.transform(test['sentence'])
print('X dims:', Xtr.shape)

# C features: multi-hot over the full idea space
mlb = MultiLabelBinarizer()
mlb.fit(list(train['ideas']) + list(test['ideas']))
Ctr = mlb.transform(list(train['ideas']))
Cte = mlb.transform(list(test['ideas']))
print('C dims:', Ctr.shape)

XCtr = hstack([Xtr, Ctr]).tocsr()
XCte = hstack([Xte, Cte]).tocsr()


X dims: (1748, 4958)
C dims: (1748, 8706)


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

def run(Xtr_f, Xte_f, name):
    lr = LogisticRegression(max_iter=1000, C=10, class_weight='balanced')
    lr.fit(Xtr_f, train['label'])
    p = lr.predict(Xte_f)
    acc = accuracy_score(test['label'], p)
    f1 = f1_score(test['label'], p, average='macro')
    print(f'{name:12s} acc={acc:.4f}  macro-F1={f1:.4f}')
    return lr, p, acc, f1

lrA, pA, accA, f1A = run(Xtr, Xte, 'A: X only')
lrB, pB, accB, f1B = run(XCtr, XCte, 'B: X + C')

print('\nDelta (B - A):  acc %+.4f   macro-F1 %+.4f' % (accB - accA, f1B - f1A))


A: X only    acc=0.6559  macro-F1=0.6581
B: X + C     acc=0.7541  macro-F1=0.7555

Delta (B - A):  acc +0.0982   macro-F1 +0.0974


In [6]:
import pandas as pd
rows = [
    ['A: X -> Y (control)', accA, f1A],
    ['B: X + C -> Y (experiment)', accB, f1B],
]
print(pd.DataFrame(rows, columns=['Condition', 'Accuracy', 'Macro-F1']).to_string(index=False))
print('\nPer-class report for B (X + C -> Y):')
print(classification_report(test['label'], pB, zero_division=0))


                 Condition  Accuracy  Macro-F1
       A: X -> Y (control)  0.655856  0.658133
B: X + C -> Y (experiment)  0.754054  0.755501

Per-class report for B (X + C -> Y):
                                                                 precision    recall  f1-score   support

                   Categories of Personal Information Collected       0.55      0.74      0.64       109
          Categories of Personal Information Shared / Disclosed       0.54      0.63      0.58       101
                        Categories of Personal Information Sold       0.63      0.65      0.64        83
                    Description of Right to Correct Information       0.88      0.87      0.88        93
                                 Description of Right to Delete       0.94      0.82      0.88        99
                      Description of Right to Know PI Collected       0.79      0.55      0.65        89
                  Description of Right to Know PI sold / shared       0.69      0.59 

## Attribution (the linear-probe payoff)

Because the head is strictly linear, per-idea → label attribution is exact:
`weight(idea, label)`. Below, the top contributing ideas per label from the
B-model's idea block.

In [7]:
# Extract the C-block weights (idea dims are the last Ctr.shape[1] columns)
W = lrB.coef_                                  # (12, n_features)
WC = W[:, -Ctr.shape[1]:]                     # (12, n_ideas)
classes = lrB.classes_
id2label_idx = {mlb.classes_[j]: j for j in range(Ctr.shape[1])}
central = ideas.drop_duplicates('idea_id').set_index('idea_id')['central'].to_dict()

for ci, c in enumerate(classes):
    top = np.argsort(WC[ci])[::-1][:3]
    print(f'\n[{c}]')
    for j in top:
        idea = mlb.classes_[j]
        print(f'   w={WC[ci][j]:+.3f}  {central.get(idea, "")[:100]}')



[Categories of Personal Information Collected]
   w=+2.517  A sentence that enumerates the types of data collected (e.g., activity, preferences, inferred attrib
   w=+1.946  A sentence that describes the use of tracking technologies that gather data about a user's online be
   w=+1.721  A label concerning the categories of personal information collected applies to any sentence that ide

[Categories of Personal Information Shared / Disclosed]
   w=+2.101  A sentence that enumerates specific circumstances or purposes under which personal data is shared wi
   w=+2.076  A sentence that identifies a third-party recipient and then enumerates the types of user data that m
   w=+1.987  A sentence that enumerates distinct types of external parties or circumstances under which personal 

[Categories of Personal Information Sold]
   w=+2.974  The label 'Categories of Personal Information Sold' applies to sentences that enumerate distinct typ
   w=+1.882  When a disclosure describes sharing perso